# Evaluation

This notebook evaluates the DPO-trained model against the base model using an LLM-as-a-judge protocol.

Rather than relying solely on perplexity or loss-based metrics, we assess:

- Coherence
- Accuracy
- Coverage
- Overall quality

Evaluation is performed using a strong external judge model to provide structured scoring and reasoning.

This allows us to measure whether preference optimization improves judgment quality.

In [ ]:
import os; os.environ.setdefault('HF_HUB_DISABLE_XET', '1')  # avoid full-repo downloads
import json
import os
from asyncio import Semaphore

import torch
from datasets import load_from_disk
from openai import AsyncOpenAI
from pydantic_settings import BaseSettings, SettingsConfigDict
from tqdm.asyncio import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from utils.evaluation_helpers import (
    extract_qa,
    judge_with_llm,
    run_local_inference,
)


os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"  # Adjust or comment out if fewer GPUs are available

In [ ]:
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    """Walk up from start until a directory containing .env is found."""
    for directory in [start, *start.parents]:
        if (directory / ".env").exists():
            return directory
    return start  # fallback: assume cwd is root


REPO_ROOT = _find_repo_root(Path.cwd())


class Config(BaseSettings):
    """Configuration settings for evaluation."""

    GEMINI_API_KEY: str = ""
    OPENAI_API_KEY: str = ""

    model_config = SettingsConfigDict(
        env_file=REPO_ROOT / ".env",
        env_file_encoding="utf-8",
        extra="ignore",
    )


config = Config()
semaphore = Semaphore(5)
print(f"Repo root: {REPO_ROOT}")
print(f"Reading .env from: {REPO_ROOT / '.env'}")

In [ ]:
from pathlib import Path


BASE_MODEL = "Qwen/Qwen2-7B-Instruct"

# ── Judge configuration ──────────────────────────────────────────────────────
# Set JUDGE_PROVIDER to "gemini" (default) or "openai".
# Gemini is used by default; OpenAI keys are not provided on this platform.
JUDGE_PROVIDER = "gemini"  # "gemini" | "openai"

if JUDGE_PROVIDER == "gemini":
    assert config.GEMINI_API_KEY, "Set GEMINI_API_KEY in your .env file"
    client = AsyncOpenAI(
        api_key=config.GEMINI_API_KEY,
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    )
    JUDGE_MODEL = "gemini-2.5-flash-lite"
elif JUDGE_PROVIDER == "openai":
    assert config.OPENAI_API_KEY, "Set OPENAI_API_KEY in your .env file"
    client = AsyncOpenAI(api_key=config.OPENAI_API_KEY)
    JUDGE_MODEL = "gpt-4o-mini"
else:
    raise ValueError(f"Unknown JUDGE_PROVIDER: {JUDGE_PROVIDER!r}")

# ─────────────────────────────────────────────────────────────────────────────
MAX_NEW_TOKENS = 2048
TEMPERATURE = 0.0
SAMPLE_LIMIT = 50
EPSILON = 1e-3

# Must match previous notebooks
DATA_FOLDER_NAME = "data_sky"  # <-- change if needed

# Resolve notebook directory the same way as 04_dpo_training.ipynb
try:
    NOTEBOOK_DIR = Path(__file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path.cwd()

# Model directory (same logic as training notebook)
MODEL_TAG = BASE_MODEL.rsplit("/", maxsplit=1)[-1].replace("-", "")
DPO_MODEL_PATH = NOTEBOOK_DIR / "models" / f"{DATA_FOLDER_NAME}_DPO_{MODEL_TAG}"

if not DPO_MODEL_PATH.exists():
    raise FileNotFoundError(f"{DPO_MODEL_PATH} not found. Please run 04_dpo_training.ipynb first.")

# DPO dataset directory
DATASET_PATH = NOTEBOOK_DIR / f"dpo_dataset_{DATA_FOLDER_NAME}"

if not DATASET_PATH.exists():
    raise FileNotFoundError(f"{DATASET_PATH} not found. Please run 03_dpo_pair_construction.ipynb first.")

# Output results file
OUTPUT_JSON = NOTEBOOK_DIR / f"llm_judge_results_{DATA_FOLDER_NAME}.json"

print(f"Judge: {JUDGE_PROVIDER} / {JUDGE_MODEL}")
print(f"Evaluating model at: {DPO_MODEL_PATH}")
print(f"Using dataset at: {DATASET_PATH}")
print(f"Results will be saved to: {OUTPUT_JSON}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa",  # Choose "flash_attention" or "sdpa" based on your GPU capabilities
).eval()

dpo_model = AutoModelForCausalLM.from_pretrained(
    DPO_MODEL_PATH,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa",  # Choose "flash_attention" or "sdpa" based on your GPU capabilities
).eval()

In [ ]:
dataset = load_from_disk(DATASET_PATH)["validation"]
dataset = dataset.select(range(min(SAMPLE_LIMIT, len(dataset))))
print(f"Evaluating {len(dataset)} validation samples")

### Extracting Structured QA Components

To ensure fair evaluation, we extract:

- ```The original question```
- ```Answer 1```
- ```Answer 2```

This allows both the base model and DPO-trained model outputs to be evaluated under identical input conditions.

Consistent input formatting is critical to prevent evaluation bias.

### Deterministic Local Inference

We generate judgments from both:

- The base model
- The DPO-trained model

using deterministic decoding (do_sample=False).
Deterministic inference reduces variance introduced by sampling, ensuring that observed differences reflect training effects rather than stochastic noise.

### LLM-as-a-Judge Evaluation Protocol

#### Why LLM-as-a-Judge?

Recent work has shown that strong LLMs can serve as reliable automated evaluators when properly prompted and controlled.

LLM-based evaluation:

- Correlates well with human judgment
- Enables scalable comparison
- Provides interpretable reasoning traces
- We use temperature = 0 during judging to ensure deterministic, stable scoring.


We evaluate model outputs using a strong external LLM acting as an impartial judge.
The judge independently scores:

- Base model judgment
- DPO-trained model judgment

on a scale of 0–10 across:

- Coherence
- Accuracy
- Coverage
- Overall quality

The judge returns structured JSON:
```bash
{
  "base_score": number,
  "dpo_score": number,
  "reason": "short explanation"
}
```


This structured evaluation provides both quantitative comparison (scores) and qualitative justification (reasoning).

Using an external judge reduces bias that may arise from self-evaluation.

In [ ]:
async def run_evaluation():
    stats = {
        "base_wins": 0,
        "dpo_wins": 0,
        "ties": 0,
        "base_scores": [],
        "dpo_scores": [],
    }

    records = []
    printed_example = False

    for sample in tqdm(dataset, desc="Evaluating"):
        conv_text = sample["conversations"][0]["value"]
        q, a1, a2 = extract_qa(conv_text)

        if not printed_example:
            print("\nQUESTION:\n", q)
            print("\nANSWER 1:\n", a1)
            print("\nANSWER 2:\n", a2)
            printed_example = True

        prompt = (
            "As an evaluation expert, given a question and its two possible answers, "
            "please select which answer best meets the criteria of coherence, accuracy, "
            "coverage, and overall quality.\n"
            "Output JSON with fields 'reason' and 'better_answer'.\n\n"
            f"Question: {q}\n"
            f"Answer 1: {a1}\n"
            f"Answer 2: {a2}\n"
        )

        base_out = run_local_inference(base_model, tokenizer, prompt, MAX_NEW_TOKENS, TEMPERATURE)
        dpo_out = run_local_inference(dpo_model, tokenizer, prompt, MAX_NEW_TOKENS, TEMPERATURE)

        judgment = await judge_with_llm(client, semaphore, JUDGE_MODEL, q, a1, a2, base_out, dpo_out)

        b = float(judgment["base_score"])
        d = float(judgment["dpo_score"])

        stats["base_scores"].append(b)
        stats["dpo_scores"].append(d)

        if abs(d - b) < EPSILON:
            stats["ties"] += 1
            winner = "tie"
        elif d > b:
            stats["dpo_wins"] += 1
            winner = "dpo"
        else:
            stats["base_wins"] += 1
            winner = "base"

        records.append(
            {
                "question": q,
                "base_score": b,
                "dpo_score": d,
                "winner": winner,
                "judge_reason": judgment["reason"],
                "base_output": base_out,
                "dpo_output": dpo_out,
            }
        )

    return stats, records

In [ ]:
stats, records = await run_evaluation()

total = stats["base_wins"] + stats["dpo_wins"]
win_rate = stats["dpo_wins"] / total if total > 0 else 0.0

print("\nRESULTS")
print(f"Base wins : {stats['base_wins']}")
print(f"DPO wins  : {stats['dpo_wins']}")
print(f"Ties      : {stats['ties']}")
print(f"DPO Win Rate : {win_rate:.3f}")
print(f"Avg Base Score : {sum(stats['base_scores']) / len(stats['base_scores']):.2f}")
print(f"Avg DPO Score  : {sum(stats['dpo_scores']) / len(stats['dpo_scores']):.2f}")

with open(OUTPUT_JSON, "w") as f:
    json.dump(records, f, indent=2)

print(f"\nSaved results to {OUTPUT_JSON}")

### References

- Zheng et al., 2023. Judging LLMs by Their Covers (MT-Bench & Chatbot Arena).
https://arxiv.org/abs/2306.05685
- Liu et al., 2023. G-Eval: NLG Evaluation using GPT-4 with Better Human Alignment.
https://arxiv.org/abs/2303.16634
- Ye et al., 2025. Learning LLM-as-a-Judge for Preference Alignment.
https://openreview.net/forum?id=HZVIQE1MsJ